### Here we are doing the initial Data Exploration. In the below code I am trying to do following things:
### 1. Read a dataset file from 2024 just to get the gist
### 2. Print its schema (data hierarchy)
### 3. Peek at the top 20 of the dataset
### 4. Count total row of that piece of dataset

In [0]:
df = spark.read.csv("s3://noaa-ghcn-pds/csv/by_year/2024.csv", header=True, inferSchema=False)

df.printSchema()

df.show(20, truncate=False)

print(df.count())

root
 |-- ID: string (nullable = true)
 |-- DATE: string (nullable = true)
 |-- ELEMENT: string (nullable = true)
 |-- DATA_VALUE: string (nullable = true)
 |-- M_FLAG: string (nullable = true)
 |-- Q_FLAG: string (nullable = true)
 |-- S_FLAG: string (nullable = true)
 |-- OBS_TIME: string (nullable = true)

+-----------+--------+-------+----------+------+------+------+--------+
|ID         |DATE    |ELEMENT|DATA_VALUE|M_FLAG|Q_FLAG|S_FLAG|OBS_TIME|
+-----------+--------+-------+----------+------+------+------+--------+
|AE000041196|20240101|TMAX   |278       |NULL  |NULL  |S     |NULL    |
|AE000041196|20240101|TMIN   |182       |NULL  |NULL  |S     |NULL    |
|AE000041196|20240101|PRCP   |0         |D     |NULL  |S     |NULL    |
|AE000041196|20240101|TAVG   |236       |H     |NULL  |S     |NULL    |
|AEM00041194|20240101|TMAX   |277       |NULL  |NULL  |S     |NULL    |
|AEM00041194|20240101|TMIN   |208       |NULL  |NULL  |S     |NULL    |
|AEM00041194|20240101|PRCP   |0         |

### We will be performing "GroupBy" element since our project topic is to find the anomaly detection. So. after pulling just ELEMENT data, we can choose which anomaly detection to focus on.

In [0]:
df.groupBy("ELEMENT").count().orderBy("count", ascending=False).show(50, truncate=False)

+-------+--------+
|ELEMENT|count   |
+-------+--------+
|PRCP   |11265765|
|SNOW   |5702213 |
|TMAX   |4109938 |
|TMIN   |4100115 |
|SNWD   |3227678 |
|TAVG   |1962030 |
|TOBS   |1568887 |
|WESD   |575493  |
|AWND   |421296  |
|WSF2   |397507  |
|WDF2   |397350  |
|WSF5   |384506  |
|WDF5   |383858  |
|WESF   |334365  |
|WT01   |166899  |
|RHMX   |165308  |
|RHMN   |165308  |
|RHAV   |165245  |
|ADPT   |164517  |
|AWBT   |164517  |
|ASTP   |164215  |
|ASLP   |164215  |
|DAPR   |135910  |
|MDPR   |134694  |
|PGTM   |112681  |
|WT03   |64078   |
|WSFG   |58360   |
|WDFG   |52276   |
|SX32   |45909   |
|SN32   |45906   |
|WT08   |43324   |
|EVAP   |39982   |
|WDMV   |31372   |
|WT02   |23651   |
|MXPN   |21815   |
|MNPN   |21534   |
|SX52   |17182   |
|SN52   |16796   |
|WSFI   |12679   |
|AWDR   |11998   |
|DWPR   |5834    |
|SN31   |5192    |
|SX31   |5189    |
|SX33   |4216    |
|SN33   |4207    |
|WT11   |3652    |
|WT06   |3080    |
|MDTX   |2853    |
|DATX   |2853    |
|MDTN   |278

### As we've decided that we'll focus only on the 3 elements TMAX(temperature max), TMIN(temperature min), PRCP(precipitation). Now we want to create a table of just those 3 elements and their count for further Data Exploration. 

In [0]:
core_elements = ["TMAX", "TMIN", "PRCP"]
df_core = df.filter(df.ELEMENT.isin(core_elements))

df_core.groupBy("ELEMENT").count().show()
print(df_core.count())


+-------+--------+
|ELEMENT|   count|
+-------+--------+
|   PRCP|11265765|
|   TMIN| 4100115|
|   TMAX| 4109938|
+-------+--------+

19475818


### Now that we have elements that we want to focus, we are going to check the quality of the data. We have a lot of the column but we will especially focus on the "DATA_VALUE" and "Q_FLAG" column right now. 
### 
### DATA_VALUE: it is the measurement itself — if it's missing or garbage, everything downstream
### Q_Flag: it is GHCN's own built-in quality-assurance flag — it tells us which observations already failed automated sanity checks

In [0]:
df_core.select("DATA_VALUE").filter(df_core.DATA_VALUE.isNull()).count()

0

In [0]:
df_core.groupBy("Q_FLAG").count().orderBy("count", ascending=False).show()

+------+--------+
|Q_FLAG|   count|
+------+--------+
|  NULL|19456005|
|     I|    6297|
|     L|    4595|
|     Z|    2793|
|     D|    1512|
|     S|    1464|
|     K|    1394|
|     O|    1179|
|     G|     363|
|     N|     118|
|     R|      46|
|     X|      25|
|     T|      21|
|     M|       6|
+------+--------+



### 19,454,812 out of 19,474,645 rows have a NULL Q_FLAG — meaning only about 19,833 rows (roughly 0.1%) failed any quality check. That's a tiny, safe amount to drop. We'll now filter those flagged rows out, keeping only clean data.

In [0]:
df_clean = df_core.filter(df_core.Q_FLAG.isNull())
print(df_clean.count())

19456005


### Now we want to cast column to proper types. Currenlty, everything is a string. So, first we will convert DATE and DATA_VALUE


In [0]:
from pyspark.sql.functions import col, to_date

df_typed = df_clean.withColumn("DATE", to_date(col("DATE"), "yyyyMMdd"))\
    .withColumn("DATA_VALUE", col("DATA_VALUE").cast("int"))

df_typed.printSchema()
df_typed.show(10, truncate=False)

root
 |-- ID: string (nullable = true)
 |-- DATE: date (nullable = true)
 |-- ELEMENT: string (nullable = true)
 |-- DATA_VALUE: integer (nullable = true)
 |-- M_FLAG: string (nullable = true)
 |-- Q_FLAG: string (nullable = true)
 |-- S_FLAG: string (nullable = true)
 |-- OBS_TIME: string (nullable = true)

+-----------+----------+-------+----------+------+------+------+--------+
|ID         |DATE      |ELEMENT|DATA_VALUE|M_FLAG|Q_FLAG|S_FLAG|OBS_TIME|
+-----------+----------+-------+----------+------+------+------+--------+
|AE000041196|2024-01-01|TMAX   |278       |NULL  |NULL  |S     |NULL    |
|AE000041196|2024-01-01|TMIN   |182       |NULL  |NULL  |S     |NULL    |
|AE000041196|2024-01-01|PRCP   |0         |D     |NULL  |S     |NULL    |
|AEM00041194|2024-01-01|TMAX   |277       |NULL  |NULL  |S     |NULL    |
|AEM00041194|2024-01-01|TMIN   |208       |NULL  |NULL  |S     |NULL    |
|AEM00041194|2024-01-01|PRCP   |0         |NULL  |NULL  |S     |NULL    |
|AEM00041217|2024-01-01|

### If we see the .README file of the source data(https://www.ncei.noaa.gov/pub/data/ghcn/daily/readme.txt), we can see that the value of PRCP is in 'tenths of mm' and TMAX/TMIN in 'tenths of degrees Celsius'. So, we make change accordingly. 

In [0]:
df_scaled = df_typed.withColumn("DATA_VALUE", col("DATA_VALUE") / 10.0)

df_scaled.show(10, truncate=False)


+-----------+----------+-------+----------+------+------+------+--------+
|ID         |DATE      |ELEMENT|DATA_VALUE|M_FLAG|Q_FLAG|S_FLAG|OBS_TIME|
+-----------+----------+-------+----------+------+------+------+--------+
|AE000041196|2024-01-01|TMAX   |27.8      |NULL  |NULL  |S     |NULL    |
|AE000041196|2024-01-01|TMIN   |18.2      |NULL  |NULL  |S     |NULL    |
|AE000041196|2024-01-01|PRCP   |0.0       |D     |NULL  |S     |NULL    |
|AEM00041194|2024-01-01|TMAX   |27.7      |NULL  |NULL  |S     |NULL    |
|AEM00041194|2024-01-01|TMIN   |20.8      |NULL  |NULL  |S     |NULL    |
|AEM00041194|2024-01-01|PRCP   |0.0       |NULL  |NULL  |S     |NULL    |
|AEM00041217|2024-01-01|TMAX   |27.1      |NULL  |NULL  |S     |NULL    |
|AEM00041217|2024-01-01|TMIN   |20.6      |NULL  |NULL  |S     |NULL    |
|AEM00041218|2024-01-01|TMAX   |27.5      |NULL  |NULL  |S     |NULL    |
|AEM00041218|2024-01-01|TMIN   |17.9      |NULL  |NULL  |S     |NULL    |
+-----------+----------+-------+------

### Now we are looking at the different metadata where we have the details of the physical location of the station. We need this because our csv data only gives the say TMAX value but the location and elevation of the station has a huge impact on this that we need to consider. In future, we might have to make necessary joins with this table so we are doing basic data exploration on this as well. 

In [0]:
stations_raw = spark.read.text("s3://noaa-ghcn-pds/ghcnd-stations.txt")
stations_raw.show(10, truncate=False)
print(stations_raw.count())

+-------------------------------------------------------------------------------------+
|value                                                                                |
+-------------------------------------------------------------------------------------+
|ACW00011604  17.1167  -61.7833   10.1    ST JOHNS COOLIDGE FLD                       |
|ACW00011647  17.1333  -61.7833   19.2    ST JOHNS                                    |
|AE000041196  25.3330   55.5170   34.0    SHARJAH INTER. AIRP            GSN     41196|
|AEM00041194  25.2550   55.3640   10.4    DUBAI INTL                             41194|
|AEM00041217  24.4330   54.6510   26.8    ABU DHABI INTL                         41217|
|AEM00041218  24.2620   55.6090  264.9    AL AIN INTL                            41218|
|AF000040930  35.3170   69.0170 3366.0    NORTH-SALANG                   GSN     40930|
|AFM00040938  34.2100   62.2280  977.2    HERAT                                  40938|
|AFM00040948  34.5660   69.2120 

### After looking at the raw lines, we looked up NOAA's official documentation to get the exact character positions (columns 1-11 = ID, 13-20 = latitude, etc.), then used .substr(start, length) to slice each line into real columns, producing your stations dataframe with proper ID, LATITUDE, LONGITUDE, ELEVATION, STATE, NAME columns.

In [0]:
from pyspark.sql.functions import trim

stations = stations_raw.select(
    trim(stations_raw.value.substr(1, 11)).alias("ID"),
    trim(stations_raw.value.substr(13, 8)).cast("double").alias("LATITUDE"),
    trim(stations_raw.value.substr(22, 9)).cast("double").alias("LONGITUDE"),
    trim(stations_raw.value.substr(32, 6)).cast("double").alias("ELEVATION"),
    trim(stations_raw.value.substr(39, 2)).alias("STATE"),
    trim(stations_raw.value.substr(42, 30)).alias("NAME")
)

stations.show(10, truncate=False)
print(stations.count())

+-----------+--------+---------+---------+-----+---------------------+
|ID         |LATITUDE|LONGITUDE|ELEVATION|STATE|NAME                 |
+-----------+--------+---------+---------+-----+---------------------+
|ACW00011604|17.1167 |-61.7833 |10.1     |     |ST JOHNS COOLIDGE FLD|
|ACW00011647|17.1333 |-61.7833 |19.2     |     |ST JOHNS             |
|AE000041196|25.333  |55.517   |34.0     |     |SHARJAH INTER. AIRP  |
|AEM00041194|25.255  |55.364   |10.4     |     |DUBAI INTL           |
|AEM00041217|24.433  |54.651   |26.8     |     |ABU DHABI INTL       |
|AEM00041218|24.262  |55.609   |264.9    |     |AL AIN INTL          |
|AF000040930|35.317  |69.017   |3366.0   |     |NORTH-SALANG         |
|AFM00040938|34.21   |62.228   |977.2    |     |HERAT                |
|AFM00040948|34.566  |69.212   |1791.3   |     |KABUL INTL           |
|AFM00040990|31.5    |65.85    |1010.0   |     |KANDAHAR AIRPORT     |
+-----------+--------+---------+---------+-----+---------------------+
only s

### Verifying the join will actually work. Before trusting that you can combine observations with station metadata later, I checked two things: that stations.ID has no duplicates, and that every station ID appearing in my observation data actually exists in the stations table. This matters because if station IDs didn't line up, our future join would silently drop data or produce nulls.

In [0]:
# check for duplicate station IDs
print(stations.count())
print(stations.select("ID").distinct().count())

# check how many of df_scaled's station IDs exist in stations
obs_ids = df_scaled.select("ID").distinct()
stations_ids = stations.select("ID").distinct()

matched = obs_ids.join(stations_ids, on="ID", how="inner").count()
print("Observation station IDs:", obs_ids.count())
print("Matched to stations table:", matched)

132503
132503
Observation station IDs: 44421
Matched to stations table: 44421


### Now for our project, I checked what years of data even exist in the S3 bucket. Then, made a scoping decision: use 2016–2025 as the 10-year baseline period.

In [0]:
years_listing = dbutils.fs.ls("s3://noaa-ghcn-pds/csv/by_year/")
for f in years_listing:
    print(f.name)

1750.csv
1763.csv
1764.csv
1765.csv
1766.csv
1767.csv
1768.csv
1769.csv
1770.csv
1771.csv
1772.csv
1773.csv
1774.csv
1775.csv
1776.csv
1777.csv
1778.csv
1779.csv
1780.csv
1781.csv
1782.csv
1783.csv
1784.csv
1785.csv
1786.csv
1787.csv
1788.csv
1789.csv
1790.csv
1791.csv
1792.csv
1793.csv
1794.csv
1795.csv
1796.csv
1797.csv
1798.csv
1799.csv
1800.csv
1801.csv
1802.csv
1803.csv
1804.csv
1805.csv
1806.csv
1807.csv
1808.csv
1809.csv
1810.csv
1811.csv
1812.csv
1813.csv
1814.csv
1815.csv
1816.csv
1817.csv
1818.csv
1819.csv
1820.csv
1821.csv
1822.csv
1823.csv
1824.csv
1825.csv
1826.csv
1827.csv
1828.csv
1829.csv
1830.csv
1831.csv
1832.csv
1833.csv
1834.csv
1835.csv
1836.csv
1837.csv
1838.csv
1839.csv
1840.csv
1841.csv
1842.csv
1843.csv
1844.csv
1845.csv
1846.csv
1847.csv
1848.csv
1849.csv
1850.csv
1851.csv
1852.csv
1853.csv
1854.csv
1855.csv
1856.csv
1857.csv
1858.csv
1859.csv
1860.csv
1861.csv
1862.csv
1863.csv
1864.csv
1865.csv
1866.csv
1867.csv
1868.csv
1869.csv
1870.csv
1871.csv
1872.csv
1

### I then tried actually reading all 11 years (2016–2026) into one dataframe as a stress test. Since I was on Databricks free edition, I wanted to confirming Free Edition compute can handle this volume.

In [0]:
years = list(range(2016, 2027))
paths = [f"s3://noaa-ghcn-pds/csv/by_year/{y}.csv" for y in years]

df_all = spark.read.csv(paths, header=True, inferSchema=False)

df_all.printSchema()
print(df_all.count())

root
 |-- ID: string (nullable = true)
 |-- DATE: string (nullable = true)
 |-- ELEMENT: string (nullable = true)
 |-- DATA_VALUE: string (nullable = true)
 |-- M_FLAG: string (nullable = true)
 |-- Q_FLAG: string (nullable = true)
 |-- S_FLAG: string (nullable = true)
 |-- OBS_TIME: string (nullable = true)

391479581
